In [ ]:
import numpy as np
from datasets import load_dataset, DatasetDict


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/src/ner') if Path('/kaggle/working/src/ner').is_dir() else Path.cwd() / 'src' / 'ner'
DATA_ROOT = PROJECT_ROOT / 'data'
CACHE_ROOT = DATA_ROOT / 'source_cache'
FEW_NERD_OUTPUT_PATH = DATA_ROOT / 'few_nerd_mountains_output'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
FEW_NERD_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

np.random.seed(42)
NUM_PROCESSES = 0
MOUNTAIN_IDX = 24
print('project_root=', PROJECT_ROOT)
print('output_path=', FEW_NERD_OUTPUT_PATH)


In [ ]:
ds = load_dataset(
    'DFKI-SLT/few-nerd',
    'supervised',
    cache_dir=str(CACHE_ROOT),
)
ds


In [ ]:
fine_labels = ds["train"].features["fine_ner_tags"].feature.names
fine_labels

In [ ]:
fine_labels[MOUNTAIN_IDX]

In [ ]:
def print_mountain_stats(dataset_split, split_name: str, mountain_idx: int = 24):
    """
    Iterates through the dataset in a single pass, collecting all necessary statistics.
    """
    mountain_count = 0
    num_mountain_samples = 0
    unique_mountains = set()

    for sample in dataset_split:
        tags = sample["fine_ner_tags"]
        if mountain_idx in tags:
            num_mountain_samples += 1
            for idx, tag in enumerate(tags):
                if tag == mountain_idx:
                    mountain_count += 1
                    unique_mountains.add(sample["tokens"][idx])

    print(f"=== Statistics for split: {split_name.upper()} ===")
    print(f"Total number of mountain tokens: {mountain_count}")
    print(f"Number of unique mountains: {len(unique_mountains)}")
    print(f"Number of sentences with mountains: {num_mountain_samples}\n")

    return {
        "num_mountain_samples": num_mountain_samples,
        "unique_mountains": unique_mountains,
        "mountain_count": mountain_count,
    }

print("\nAnalyzing initial class distribution:\n" + "-" * 40)
train_stats = print_mountain_stats(ds["train"], "train", MOUNTAIN_IDX)
val_stats = print_mountain_stats(ds["validation"], "validation", MOUNTAIN_IDX)
test_stats = print_mountain_stats(ds["test"], "test", MOUNTAIN_IDX)

In [ ]:
def process_example(example):
    """
    Converts multi-class labels into binary BIO format for mountains (O, B-Mountain, I-Mountain)
    and joins tokens into a single sentence string.
    """
    new_tags = []
    found_mountain = False

    for tag in example["fine_ner_tags"]:
        if tag == MOUNTAIN_IDX:
            if not found_mountain:
                new_tags.append(1)  # B-Mountain -> 1
                found_mountain = True
            else:
                new_tags.append(2)  # I-Mountain -> 2
        else:
            new_tags.append(0)      # O -> 0
            found_mountain = False

    example["labels"] = new_tags
    example["sentence"] = " ".join(example["tokens"])
    return example

print("Relabeling tags and formatting columns (multiprocessing)...")
# Apply mapping and immediately remove unnecessary original columns
ds_processed = ds.map(
    process_example,
    num_proc=NUM_PROCESSES,
    remove_columns=["ner_tags", "fine_ner_tags"]
)
ds_processed['train'][0]

In [ ]:
def balance_split(dataset, stats, split_name):
    """
    Reduces the number of sentences without mountains (downsampling) to balance the classes.
    """
    mountain_samples_count = stats["num_mountain_samples"]
    total_samples = len(dataset)
    non_mountain_samples = total_samples - mountain_samples_count

    # Calculate the keep probability for empty sentences to achieve a ~50/50 ratio
    p = mountain_samples_count / non_mountain_samples if non_mountain_samples > 0 else 1.0

    def keep_or_discard(example):
        # 1 is the B-Mountain tag (presence of the entity in the sentence)
        has_mountain = 1 in example["labels"]
        return has_mountain or np.random.rand() < p

    balanced = dataset.filter(keep_or_discard, num_proc=NUM_PROCESSES)

    print(f"[{split_name.upper()}] Size reduced from {total_samples} to {len(balanced)} samples (p_keep = {p:.4f})")
    return balanced

print("\nBalancing splits...")
balanced_ds = DatasetDict({
    "train": balance_split(ds_processed["train"], train_stats, "train"),
    "validation": balance_split(ds_processed["validation"], val_stats, "validation"),
    "test": balance_split(ds_processed["test"], test_stats, "test")
})


print(f"\nSaving balanced dataset to disk: {FEW_NERD_OUTPUT_PATH}...")
balanced_ds.save_to_disk(FEW_NERD_OUTPUT_PATH)

print("\nDone! Pipeline executed successfully.")